In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
pip install langchainhub

  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)

  Attempting uninstall: packaging

    Found existing installation: packaging 25.0

    Uninstalling packaging-25.0:

      Successfully uninstalled packaging-25.0

   ------------- -------------------------- 1/3 [packaging]
   ------------- -------------------------- 1/3 [packaging]
   ------------- -------------------------- 1/3 [packaging]
   ------------- -------------------------- 1/3 [packaging]
   -------------------------- ------------- 2/3 [langchainhub]
   ---------------------------------------- 3/3 [langchainhub]

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.0.2 requires pydantic<=2.12.4,>=2.11.10, but you have pydantic 2.12.5 which is incompatible.


In [13]:
from langsmith import Client
from langchain_core.prompts import PromptTemplate

d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
client = Client()
react_prompt = client.pull_prompt("hwchase17/react")

In [27]:
react_prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [28]:
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

@tool
def triple(num: float) -> float:
    """
    :param num: a number to triple
    :return: the number tripled -> multiplied by 3
    """

    print("================executing toll triple with input=",num,"===")
    return 3 * float(num)

tools = [triple, TavilySearchResults(max_results=1)]

In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

In [51]:
from langchain_classic.agents import create_react_agent
from langchain_classic.agents import AgentExecutor

react_agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt
)

agent_executor = AgentExecutor(
    agent=react_agent,
    tools=tools,
    verbose=True
)

In [ ]:
react_agent.invoke(
    {
        "input": "What is the triple of the current population of New York City?",
        "intermediate_steps": []
    }
)

In [ ]:
from typing import TypedDict,Union, Annotated
import operator

class AgentState(TypedDict):
    input: str
    agent_outcome: Union['AgentAction', 'AgentFinish',None]
    intermediate_steps: Annotated[list[tuple[AgentAction,str]], operator.add]

In [ ]:
def run_agent_reasoning_engine(state: AgentState):
    agent_outcome=react_agent.invoke(state)
    
    return {
        "agent_outcome": agent_outcome
    }

In [ ]:
from langgrpah.graph import END

def should_continue(state:AgentState)->str:
    print("Inside should continue")

    if isinstance(state["agent_outcome"], AgentFinish):
        return END
    else:
        return "ac"

In [ ]:
from langgraph.prebuilt.tool_executor import ToolExecutor

# agent_action = react_agent.invoke(
#     {
#         "input": "What is the triple of the current population of New York City?",
#         "intermediate_steps": []
#     }
# )

tool_executor=ToolExecutor(tools)
# result=tool_executor.invoke(agent_action)
# result

In [ ]:
def execute_tools(state:AgentState):
    agent_action=state["agent_outcome"]
    output=tool_executor.invoke(agent_action)
    
    return {"intermediate_steps"[(agent_action, str(output))]}

In [ ]:
from langgraph.graph import StateGraph

flow = StateGraph(AgentState)
flow.add_node("reasoning", run_agent_reasoning_engine)
flow.set_entry_node("reasoning")
flow.add_node("ac", execute_tools)

flow.add_conditional_edges(
    "reasoning",should_continue, {
        "act":tool_executor,
        "END":"END"
    }
)

flow.add_edge("tool_executor","reasoning")

app=flow.compile()